# MCP audio : exposer la transcription, la synthèse et l'analyse comme outils d'agent

**Module :** 03-Audio-Orchestration
**Niveau :** Avancé
**Technologies :** Model Context Protocol (FastMCP), faster-whisper, Kokoro ONNX, OpenAI-compatible (vLLM)
**Durée estimée :** 45 minutes

## Objectifs d'apprentissage

- [ ] Exposer des capacités audio réelles comme outils du protocole MCP (FastMCP, SDK officiel)
- [ ] Interroger ce serveur depuis un client MCP réel : `initialize`, `tools/list`, `tools/call`
- [ ] Faire choisir les outils par un **modèle réel** qui reçoit une consigne en langue naturelle
- [ ] Mesurer le choix d'outil, la latence par appel et le temps bout en bout, puis lire ces chiffres
- [ ] Distinguer la latence de l'agent (modèle) de la latence de l'outil (moteur audio)

## Prérequis

- Notebooks [01-4](https://github.com/jsboige/CoursIA/blob/main/MyIA.AI.Notebooks/GenAI/Audio/01-Foundation/01-4-Whisper-Local.ipynb) (Whisper local) et [01-5](https://github.com/jsboige/CoursIA/blob/main/MyIA.AI.Notebooks/GenAI/Audio/01-Foundation/01-5-Kokoro-TTS-Local.ipynb) (Kokoro) : ce carnet **réutilise** leurs moteurs, il n'en réimplémente aucun.
- Un moteur de chat compatible OpenAI avec appel d'outils : ici le vLLM local du cluster (`http://127.0.0.1:5002/v1`). La clé est lue dans `GenAI/.env` (variable `VLLM_API_KEY`).
- `pip install mcp faster-whisper kokoro-onnx openai` — l'environnement de la série (`coursia-genai-audio`) les fournit.
- Aucun GPU requis : les trois moteurs tournent sur CPU.

**Navigation :** [<< 03-3](03-3-Realtime-Voice-API.ipynb) | [Index](../README.md) | [Index Audio](../README.md)

In [1]:
# Parametres Papermill - JAMAIS modifier ce commentaire

# Configuration notebook
notebook_mode = "batch"              # "interactive" ou "batch"
skip_widgets = True                  # True pour mode batch MCP
debug_level = "INFO"

# Moteurs audio de la serie (CPU)
whisper_model = "small"              # taille faster-whisper
whisper_compute = "int8"             # type de calcul ctranslate2
kokoro_voice = "af_heart"            # voix Kokoro (anglais)
kokoro_voice_fr = "ff_siwis"         # voix Kokoro (francais)

# Agent : moteur de chat compatible OpenAI, avec appel d'outils
vllm_base_url = "http://127.0.0.1:5002/v1"
vllm_model = "qwen3.6-35b-a3b"

# Batterie de mesure
nb_taches = 7                        # nombre de consignes donnees a l'agent
nb_repetitions = 2                   # repetitions par consigne
temperature_agent = 0.2              # temperature de l'agent
max_tours_agent = 4                  # nombre maximal de tours (appels d'outils) par consigne

## 1. Environnement

Le carnet travaille dans le dépôt `CoursIA` : il localise la racine `GenAI` en remontant l'arborescence, charge le `.env` de la série (clé du moteur de chat) et prépare un dossier de sorties **hors du dépôt suivi par git**.

Les moteurs ne sont pas réinventés : `faster-whisper` et `kokoro-onnx` sont ceux des carnets [01-4](../../01-Foundation/01-4-Whisper-Local.ipynb) et [01-5](../../01-Foundation/01-5-Kokoro-TTS-Local.ipynb) ; le client MCP est le SDK officiel (`mcp`).

In [2]:
# Setup environnement et imports
import asyncio
import json
import os
import platform
import sys
import tempfile
import threading
import time
from datetime import datetime
from pathlib import Path

import requests
from dotenv import load_dotenv

# --- Racine GenAI : on remonte jusqu'au dossier qui porte le .env de la serie
GENAI_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if p.name == "GenAI"), Path.cwd())

# --- Charge du .env : la cle du moteur de chat y vit (jamais dans le carnet)
env_charges = []
candidat = GENAI_ROOT / ".env"
if candidat.exists():
    load_dotenv(candidat)
    env_charges.append(str(candidat))
else:
    print(f"WARNING : .env absent ({candidat}) — la cle du moteur de chat ne sera pas lue")

# --- Dossier de sorties (hors git : MyIA.AI.Notebooks/GenAI/outputs est ignore)
OUTPUT_DIR = GENAI_ROOT / "outputs" / "audio" / "mcp"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Repertoire des modeles Kokoro (cache partage de la serie)
CACHE_KOKORO = GENAI_ROOT / ".cache" / "kokoro-onnx"
MODELE_KOKORO = CACHE_KOKORO / "kokoro-v1.0.onnx"
VOIX_KOKORO = CACHE_KOKORO / "voices-v1.0.bin"

# --- Serveur MCP audio (livre avec le carnet, demarre en stdio par la suite)
# Le chemin est derive de la racine GenAI : il ne depend ni du repertoire
# courant du noyau, ni du nom du fichier (inconnu dans un carnet).
SERVEUR_MCP = GENAI_ROOT / "Audio" / "03-Orchestration" / "serveur_mcp_audio.py"
if not SERVEUR_MCP.exists():
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidat = base / "serveur_mcp_audio.py"
        if candidat.exists():
            SERVEUR_MCP = candidat
            break

print("MCP audio : transcription, synthese vocale et analyse exposees comme outils")
print(f"Date      : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Mode      : {notebook_mode}")
print(f"Python    : {platform.python_version()} ({sys.executable})")
print(f"GenAI     : {GENAI_ROOT}")
print(f".env      : {env_charges or '(aucun)'}")
print(f"Sorties   : {OUTPUT_DIR}")
print(f"Serveur   : {SERVEUR_MCP.name} ({'present' if SERVEUR_MCP.exists() else 'ABSENT'})")

MCP audio : transcription, synthese vocale et analyse exposees comme outils
Date      : 2026-11-14 13:18:41
Mode      : batch
Python    : 3.13.16 (<USER_PATH>\miniconda3\envs\coursia-genai-audio\python.exe)
GenAI     : D:\CoursIA-wt\audio-mcp\MyIA.AI.Notebooks\GenAI
.env      : ['D:\\CoursIA-wt\\audio-mcp\\MyIA.AI.Notebooks\\GenAI\\.env']
Sorties   : D:\CoursIA-wt\audio-mcp\MyIA.AI.Notebooks\GenAI\outputs\audio\mcp
Serveur   : serveur_mcp_audio.py (present)


### Ce que fait la machine, mesuré

Avant toute mesure d'agent, on vérifie **ce qui est réellement disponible** : versions des bibliothèques, présence des poids Kokoro, et joignabilité du moteur de chat. Un carnet qui annonce un moteur absent produit des sorties vides — cette cellule évite ce piège en nommant l'absence.

In [3]:
# Versions des briques et joignabilite du moteur de chat
from importlib.metadata import PackageNotFoundError, version

PACKAGES = ["mcp", "faster-whisper", "ctranslate2", "kokoro-onnx", "onnxruntime",
            "soundfile", "numpy", "openai"]
print("BIBLIOTHEQUES")
for paquet in PACKAGES:
    try:
        installe = version(paquet)
    except PackageNotFoundError:
        installe = "ABSENTE"
    print(f"  {paquet:<16} {installe}")

print("\nPOIDS KOKORO")
for chemin in (MODELE_KOKORO, VOIX_KOKORO):
    if chemin.exists():
        print(f"  {chemin.name:<20} {chemin.stat().st_size / (1024 * 1024):8.1f} Mo")
    else:
        print(f"  {chemin.name:<20} ABSENT ({chemin.parent})")

print("\nMOTEUR DE CHAT (compatible OpenAI, appel d'outils)")
cle_vllm = os.environ.get("VLLM_API_KEY", "")
base_vllm = os.environ.get("VLLM_BASE_URL", vllm_base_url).rstrip("/")
modeles_disponibles = []
if cle_vllm:
    try:
        reponse = requests.get(f"{base_vllm}/models",
                               headers={"Authorization": f"Bearer {cle_vllm}"}, timeout=20)
        reponse.raise_for_status()
        modeles_disponibles = [m["id"] for m in reponse.json().get("data", [])]
    except Exception as exc:
        print(f"  MOTEUR INJOIGNABLE ({type(exc).__name__} : {exc})")
else:
    print("  VLLM_API_KEY absente du .env : le moteur de chat ne peut pas etre appele")

print(f"  base    : {base_vllm}")
print(f"  modele  : {vllm_model}"
      + ("" if vllm_model in modeles_disponibles else "  (ABSENT du service)"))
print(f"  service : {modeles_disponibles or '(injoignable)'}")
moteur_chat_joignable = bool(cle_vllm and vllm_model in modeles_disponibles)

BIBLIOTHEQUES
  mcp              1.28.1
  faster-whisper   1.2.1
  ctranslate2      4.8.4
  kokoro-onnx      0.6.1
  onnxruntime      1.31.1
  soundfile        0.14.0
  numpy            2.5.3
  openai           3.35.1

POIDS KOKORO
  kokoro-v1.0.onnx        310.5 Mo
  voices-v1.0.bin          26.9 Mo

MOTEUR DE CHAT (compatible OpenAI, appel d'outils)
  base    : http://127.0.0.1:5002/v1
  modele  : qwen3.6-35b-a3b
  service : ['qwen3.6-35b-a3b', 'swift-1.5-27b']


## 2. Un audio de travail, fabriqué par un vrai moteur

Le carnet n'embarque aucun fichier audio : il **fabrique son propre support** avec Kokoro (le moteur de [01-5](../../01-Foundation/01-5-Kokoro-TTS-Local.ipynb)). Le texte de départ est conservé : il sert de **référence** pour vérifier que la transcription renvoie bien le texte réellement prononcé, et non une sortie plausible.

In [4]:
# Fabrication de l'audio de travail avec Kokoro (moteur reel, CPU)
import numpy as np
import soundfile as sf
from kokoro_onnx import Kokoro

TEXTE_REFERENCE = ("Bonjour, ceci est un test de transcription audio. "
                   "Le serveur MCP expose trois outils : transcrire, synthetiser et analyser.")
VOIX_TRAVAIL = kokoro_voice_fr
CHEMIN_AUDIO = OUTPUT_DIR / "audio_travail.wav"

if not MODELE_KOKORO.exists() or not VOIX_KOKORO.exists():
    print(f"Modele Kokoro absent de {CACHE_KOKORO} : telechargement des poids.")
    CACHE_KOKORO.mkdir(parents=True, exist_ok=True)
    for url, cible in (
        ("https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/kokoro-v1.0.onnx", MODELE_KOKORO),
        ("https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/voices-v1.0.bin", VOIX_KOKORO),
    ):
        if not cible.exists():
            reponse = requests.get(url, stream=True, timeout=300)
            reponse.raise_for_status()
            with open(cible, "wb") as f:
                for morceau in reponse.iter_content(chunk_size=1 << 20):
                    if morceau:
                        f.write(morceau)
            print(f"  {cible.name} : {cible.stat().st_size / (1024 * 1024):.1f} Mo")

kokoro = Kokoro(str(MODELE_KOKORO), str(VOIX_KOKORO))
debut_synthese = time.perf_counter()
echantillons, frequence = kokoro.create(TEXTE_REFERENCE, voice=VOIX_TRAVAIL, lang="fr-fr", speed=1.0)
temps_synthese = time.perf_counter() - debut_synthese
sf.write(str(CHEMIN_AUDIO), echantillons, frequence)

duree_audio = len(echantillons) / frequence
print(f"Voix        : {VOIX_TRAVAIL}")
print(f"Texte       : {TEXTE_REFERENCE}")
print(f"Fichier     : {CHEMIN_AUDIO.name} ({duree_audio:.2f} s, {frequence} Hz)")
print(f"Temps synthese (Kokoro, CPU) : {temps_synthese:.2f} s")
print(f"Facteur temps reel / temps calcul : {duree_audio / temps_synthese:.1f}x")

Voix        : ff_siwis
Texte       : Bonjour, ceci est un test de transcription audio. Le serveur MCP expose trois outils : transcrire, synthetiser et analyser.
Fichier     : audio_travail.wav (8.73 s, 24000 Hz)
Temps synthese (Kokoro, CPU) : 4.16 s
Facteur temps reel / temps calcul : 2.1x


## 3. Le serveur MCP audio

Le serveur [`serveur_mcp_audio.py`](serveur_mcp_audio.py) accompagne ce carnet : il expose les trois capacités audio de la série comme outils MCP (SDK officiel, `FastMCP`).

| Outil | Moteur réel appelé | Ce qu'il rend |
|---|---|---|
| `transcrire_audio` | faster-whisper (`small`, CPU int8) | le texte, la langue, la durée |
| `synthetiser_voix` | Kokoro (backend ONNX) | le chemin du WAV, sa durée, sa fréquence |
| `analyser_audio` | soundfile + numpy | durée, RMS, crête, plus long silence |

Le serveur est lancé **en stdio** : c'est le transport MCP de référence, et le client parle au serveur exactement comme le ferait un éditeur ou une plateforme d'agent.

Deux détails ne sont pas optionnels :

- **stdout appartient au protocole.** Tout journal du serveur part sur stderr ; c'est ce que le client récupère dans un fichier journal.
- **Le noyau Jupyter tient déjà une boucle d'événements.** Une session `stdio_client` ouverte dans une cellule ne survit pas à la cellule suivante : on la tient donc dans un fil dédié, avec sa propre boucle, et on lui parle par des coroutines soumises à cette boucle.

In [5]:
# Une session MCP stdio maintenue ouverte, dans un fil dedie
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


class SessionMCPPersistante:
    """Session MCP (stdio) maintenue ouverte pendant tout le carnet.

    Le noyau Jupyter fait tourner une boucle d'evenements permanente : un
    `async with stdio_client(...)` ouvert dans une cellule ne peut pas
    survivre d'une cellule a l'autre. On tient donc la session dans un fil
    dedie, avec sa propre boucle (Proactor sur Windows, pour le
    sous-processus), et on lui parle par des coroutines soumises a cette
    boucle.
    """

    def __init__(self, commande: str, args: list, env: dict) -> None:
        self.commande = commande
        self.args = list(args)
        self.env = dict(env)
        self.info = None
        self.outils: list = []
        self._session_mcp = None
        self._erreur = None
        self._pret = threading.Event()
        self.journal = open(Path(tempfile.gettempdir()) / "mcp_audio_carnet.log",
                            "w", encoding="utf-8")
        self._boucle = None
        self._fil = threading.Thread(target=self._tourner, daemon=True)
        self._fil.start()
        if not self._pret.wait(timeout=180):
            raise TimeoutError("le serveur MCP n'a pas repondu a l'initialisation en 180 s")
        if self._erreur is not None:
            raise self._erreur

    def _tourner(self) -> None:
        if sys.platform == "win32":
            self._boucle = asyncio.ProactorEventLoop()
        else:
            self._boucle = asyncio.new_event_loop()
        asyncio.set_event_loop(self._boucle)
        try:
            self._boucle.run_until_complete(self._session())
        except RuntimeError:
            pass  # boucle arretee volontairement par fermer()

    async def _session(self) -> None:
        parametres = StdioServerParameters(command=self.commande, args=self.args, env=self.env)
        try:
            async with stdio_client(parametres, errlog=self.journal) as (lecture, ecriture):
                async with ClientSession(lecture, ecriture) as session:
                    self.info = await session.initialize()
                    listing = await session.list_tools()
                    self.outils = []
                    for outil in listing.tools:
                        schema = outil.inputSchema if isinstance(outil.inputSchema, dict) \
                            else outil.inputSchema.model_dump()
                        self.outils.append({"name": outil.name,
                                            "description": outil.description or "",
                                            "inputSchema": schema})
                    self._session_mcp = session
                    self._pret.set()
                    while True:
                        await asyncio.sleep(0.2)
        except BaseException as exc:  # pragma: no cover - trace de diagnostic
            self._erreur = exc
            self._pret.set()
            raise

    def appeler(self, nom: str, arguments: dict, timeout: int = 900):
        """Appelle un outil du serveur et rend son resultat brut."""
        async def _appel():
            return await self._session_mcp.call_tool(nom, arguments)

        futur = asyncio.run_coroutine_threadsafe(_appel(), self._boucle)
        return futur.result(timeout=timeout)

    def fermer(self) -> None:
        if self._boucle is not None:
            self._boucle.call_soon_threadsafe(self._boucle.stop)
        try:
            self.journal.close()
        except Exception:
            pass


# --- Demarrage reel du serveur : le handshake est visible dans les sorties
env_serveur = {**os.environ,
               "KOKORO_MODEL_DIR": str(CACHE_KOKORO),
               "AUDIO_WORK_DIR": str(OUTPUT_DIR),
               "WHISPER_MODEL": whisper_model,
               "WHISPER_COMPUTE": whisper_compute}
session_mcp = SessionMCPPersistante(sys.executable, [str(SERVEUR_MCP)], env_serveur)

print("HANDSHAKE MCP")
print(f"  serveur           : {session_mcp.info.serverInfo.name}")
print(f"  version serveur   : {session_mcp.info.serverInfo.version}")
print(f"  version protocole : {session_mcp.info.protocolVersion}")
print(f"  journal stderr    : {session_mcp.journal.name}")
print(f"\nOUTILS EXPOSES (tools/list) : {len(session_mcp.outils)}")
for outil in session_mcp.outils:
    proprietes = ", ".join(sorted(outil["inputSchema"].get("properties", {})))
    print(f"  {outil['name']:<20} parametres : {proprietes}")
    print(f"  {'':<20} description : {outil['description'].splitlines()[0][:88]}")

HANDSHAKE MCP
  serveur           : serveur-mcp-audio
  version serveur   : 1.28.1
  version protocole : 2025-11-25
  journal stderr    : <USER_PATH>\AppData\Local\Temp\mcp_audio_carnet.log

OUTILS EXPOSES (tools/list) : 3
  transcrire_audio     parametres : chemin, langue
                       description : Transcrit un fichier audio WAV en texte (faster-whisper, CPU).
  synthetiser_voix     parametres : langue, nom_fichier, texte, vitesse, voix
                       description : Synthetise un texte en voix (Kokoro ONNX) et ecrit un fichier WAV.
  analyser_audio       parametres : chemin
                       description : Mesure un fichier audio : duree, niveau, crête et plus long silence.


### 3.1 Les trois outils, appelés pour de vrai

Un `tools/call` ne renvoie pas une chaîne de démonstration : chaque outil appelle son moteur. Le serveur **annonce lui-même** le moteur qui a produit le résultat (`moteur` dans la charge utile) et le temps qu'il a mis : c'est la preuve que l'appel est passé par l'outil et non par une réponse fabriquée.

La transcription est confrontée à la **référence** du paragraphe 2 : si le texte rendu ne contient pas cette référence, l'audio ou le moteur a changé de comportement, et le carnet le dit.

In [6]:
# Trois appels d'outils, avec confrontation a la reference
resultats_outils = {}

print("OUTIL 1/3 : transcrire_audio")
resultat_transcription = session_mcp.appeler(
    "transcrire_audio", {"chemin": str(CHEMIN_AUDIO)})
charge_transcription = json.loads(resultat_transcription.content[0].text)
resultats_outils["transcrire_audio"] = charge_transcription
print(f"  texte     : {charge_transcription['texte']}")
print(f"  langue    : {charge_transcription['langue']} "
      f"(p = {charge_transcription['probabilite_langue']})")
print(f"  duree     : {charge_transcription['duree_audio_s']} s")
print(f"  moteur    : {charge_transcription['moteur']} — {charge_transcription['temps_s']} s")
mots_reference = TEXTE_REFERENCE.split()
mots_transcrits = charge_transcription["texte"].replace(",", " ").replace(".", " ").split()
rappel_reference = [m for m in mots_reference if m.lower() in [t.lower() for t in mots_transcrits]]
print(f"  rappel de la reference : {len(rappel_reference)}/{len(mots_reference)} mots "
      f"({100 * len(rappel_reference) / len(mots_reference):.0f} %)")

print("\nOUTIL 2/3 : analyser_audio")
resultat_analyse = session_mcp.appeler(
    "analyser_audio", {"chemin": str(CHEMIN_AUDIO)})
charge_analyse = json.loads(resultat_analyse.content[0].text)
resultats_outils["analyser_audio"] = charge_analyse
print(f"  duree        : {charge_analyse['duree_s']} s a {charge_analyse['frequence']} Hz")
print(f"  RMS          : {charge_analyse['rms_dbfs']} dBFS")
print(f"  crete        : {charge_analyse['crete_dbfs']} dBFS")
print(f"  silence max  : {charge_analyse['silence_max_ms']} ms")
print(f"  temps de parole : {100 * charge_analyse['taux_parole']:.0f} %")
print(f"  moteur       : {charge_analyse['moteur']} — {charge_analyse['temps_s']} s")

print("\nOUTIL 3/3 : synthetiser_voix")
resultat_synthese = session_mcp.appeler(
    "synthetiser_voix", {"texte": "Le serveur MCP repond present.",
                          "voix": kokoro_voice, "langue": "en-us",
                          "nom_fichier": "mcp_synthese"})
charge_synthese = json.loads(resultat_synthese.content[0].text)
resultats_outils["synthetiser_voix"] = charge_synthese
print(f"  fichier   : {Path(charge_synthese['chemin']).name}")
print(f"  duree     : {charge_synthese['duree_audio_s']} s a {charge_synthese['frequence']} Hz")
print(f"  moteur    : {charge_synthese['moteur']} — {charge_synthese['temps_s']} s")

print("\nMOTIFS DE PREUVE : chaque charge utile porte le nom du moteur qui l'a produite")
for nom_outil, charge in resultats_outils.items():
    print(f"  {nom_outil:<20} -> {charge['moteur']}")

OUTIL 1/3 : transcrire_audio


  texte     : Bonjour, ceci est un test de transcription audio. Le serveur MCP expose trois outils, « Transcrire » et « Synthe size of the analyser ».
  langue    : fr (p = 0.9718)
  duree     : 8.725 s
  moteur    : faster-whisper — 6.763 s
  rappel de la reference : 13/19 mots (68 %)

OUTIL 2/3 : analyser_audio
  duree        : 8.725 s a 24000 Hz
  RMS          : -19.68 dBFS
  crete        : -1.84 dBFS
  silence max  : 310 ms
  temps de parole : 72 %
  moteur       : soundfile+numpy — 0.002 s

OUTIL 3/3 : synthetiser_voix


  fichier   : mcp_synthese.wav
  duree     : 2.347 s a 24000 Hz
  moteur    : kokoro-onnx — 3.655 s

MOTIFS DE PREUVE : chaque charge utile porte le nom du moteur qui l'a produite
  transcrire_audio     -> faster-whisper
  analyser_audio       -> soundfile+numpy
  synthetiser_voix     -> kokoro-onnx


### 3.2 Le même moteur, appelé directement

Une question légitime vient au lecteur : « l'outil MCP ajoute-t-il quelque chose, ou rejoue-t-il l'appel Python que je pourrais écrire dans une cellule ? ».

La réponse se mesure, mais il faut choisir **où** mesurer.

Le coût du protocole (aller-retour stdio, encodage JSON-RPC, thread du serveur) se mesure sur l'outil **le moins cher** — `analyser_audio`, dont le calcul prend quelques millisecondes : là, le transport est lisible. Sur un moteur lourd comme la transcription, le même coût serait noyé dans le calcul, et la différence observée entre un appel direct et un appel par l'outil refléterait surtout la variation du moteur d'un appel à l'autre — sur une machine partagée, cette variation dépasse largement le coût du protocole.

Le carnet fait donc les deux mesures, et dit laquelle conclu sur quoi.

In [7]:
# Le meme moteur (faster-whisper), appele directement dans le carnet
from faster_whisper import WhisperModel

# --- 1) Le cout du protocole, mesure sur l'outil le MOINS cher -------------
# `analyser_audio` ne demande que quelques millisecondes de calcul : ce que
# l'appel MCP ajoute (aller-retour stdio, encodage JSON-RPC, thread du
# serveur) y est lisible. Sur un moteur lourd, le protocole serait noyé dans
# le calcul : la mesure serait fausse.
REPETITIONS_PROTOCOL = 5
couts_client, couts_moteur = [], []
for _ in range(REPETITIONS_PROTOCOL):
    debut_mesure = time.perf_counter()
    reponse = session_mcp.appeler("analyser_audio", {"chemin": str(CHEMIN_AUDIO)})
    couts_client.append(time.perf_counter() - debut_mesure)
    couts_moteur.append(json.loads(reponse.content[0].text)["temps_s"])
cout_client = sorted(couts_client)[len(couts_client) // 2]
cout_moteur = sorted(couts_moteur)[len(couts_moteur) // 2]
cout_protocole = cout_client - cout_moteur

print("COUT DU PROTOCOLE (outil analyser_audio, moteur de quelques ms)")
print(f"  aller-retour client (mediane sur {REPETITIONS_PROTOCOL} appels) : {cout_client * 1000:.1f} ms")
print(f"  temps moteur annonce par l'outil                              : {cout_moteur * 1000:.1f} ms")
print(f"  cout du transport MCP (stdio + JSON-RPC)                      : {cout_protocole * 1000:.1f} ms")

# --- 2) Le meme moteur des deux cotes -------------------------------------
# Le moteur est charge une fois de chaque cote ; les deux appels sont faits
# a chaud. L'ecart entre les deux temps N'EST PAS attribuable au protocole :
# sur une machine partagee, le moteur lui-meme varie d'un appel a l'autre.
# C'est la raison de la mesure 1, qui isole le transport.
debut_direct = time.perf_counter()
modele_whisper = WhisperModel(whisper_model, device="cpu", compute_type=whisper_compute)
temps_chargement_whisper = time.perf_counter() - debut_direct

modele_whisper.transcribe(str(CHEMIN_AUDIO), beam_size=5)
debut_direct = time.perf_counter()
segments_directs, info_direct = modele_whisper.transcribe(str(CHEMIN_AUDIO), beam_size=5)
texte_direct = " ".join(s.text.strip() for s in segments_directs).strip()
temps_direct = time.perf_counter() - debut_direct

session_mcp.appeler("transcrire_audio", {"chemin": str(CHEMIN_AUDIO)})
debut_mcp = time.perf_counter()
resultat_mcp = session_mcp.appeler("transcrire_audio", {"chemin": str(CHEMIN_AUDIO)})
temps_mcp = time.perf_counter() - debut_mcp
charge_mcp = json.loads(resultat_mcp.content[0].text)

print(f"\nLE MEME MOTEUR, DEUX CHEMINS (faster-whisper '{whisper_model}', {whisper_compute}, CPU)")
print(f"  chargement      : {temps_chargement_whisper:.2f} s (exclu de la mesure)")
print(f"  appel direct    : {temps_direct:.2f} s")
print(f"  appel via l'outil MCP : {temps_mcp:.2f} s")
print(f"  ecart           : {temps_mcp - temps_direct:+.2f} s — variation du moteur, "
      f"pas un cout du protocole (voir la mesure 1)")
print(f"  texte direct    : {texte_direct}")
print(f"  texte via l'outil : {charge_mcp['texte']}")
print(f"  langue          : {info_direct.language} (p = {info_direct.language_probability:.3f})")
meme_texte = texte_direct.lower() == charge_mcp["texte"].lower()
print(f"  meme texte des deux cotes : {meme_texte}")

COUT DU PROTOCOLE (outil analyser_audio, moteur de quelques ms)
  aller-retour client (mediane sur 5 appels) : 8.7 ms
  temps moteur annonce par l'outil                              : 2.0 ms
  cout du transport MCP (stdio + JSON-RPC)                      : 6.7 ms



LE MEME MOTEUR, DEUX CHEMINS (faster-whisper 'small', int8, CPU)
  chargement      : 1.97 s (exclu de la mesure)
  appel direct    : 4.42 s
  appel via l'outil MCP : 5.45 s
  ecart           : +1.03 s — variation du moteur, pas un cout du protocole (voir la mesure 1)
  texte direct    : Bonjour, ceci est un test de transcription audio. Le serveur MCP expose trois outils, Transcrire & Synthetize of Les Analysés.
  texte via l'outil : Bonjour, ceci est un test de transcription audio. Le serveur MCP expose trois outils, « Transcrire » et « Synthe size of the analyser ».
  langue          : fr (p = 0.979)
  meme texte des deux cotes : False


## 4. L'agent qui choisit ses outils

Jusqu'ici, c'est le carnet qui appelle les outils. La question de l'issue est autre : **un agent sait-il choisir seul** ?

Le protocole :

1. le serveur expose ses outils au format MCP (`tools/list`) ;
2. ces schémas sont convertis au format d'outil d'`chat/completions` — la conversion est **une simple reprise** des schémas négociés par le serveur, pas une reconstruction ;
3. l'agent reçoit une consigne en langue naturelle et choisit (`tool_choice: "auto"`) ;
4. le carnet exécute l'outil choisi **par le canal MCP**, puis renvoie le résultat à l'agent, qui peut demander un second outil (enchaînement) ;
5. l'agent répond en texte une fois qu'il a de quoi répondre.

Le moteur est nommé : c'est le vLLM local du cluster, modèle `qwen3.6-35b-a3b`, servi sur `http://127.0.0.1:5002/v1` avec la clé lue dans `GenAI/.env`.

In [8]:
# Catalogue d'outils : des schemas MCP repris tels quels, au format OpenAI
LIMITE_DESCRIPTION = 300

catalogue_outils = []
for outil in session_mcp.outils:
    catalogue_outils.append({
        "type": "function",
        "function": {
            "name": outil["name"],
            "description": (outil["description"] or "")[:LIMITE_DESCRIPTION],
            "parameters": outil["inputSchema"] or {"type": "object", "properties": {}},
        },
    })

print("CATALOGUE D'OUTILS TRANSMIS A L'AGENT")
for fiche in catalogue_outils:
    proprietes = list(fiche["function"]["parameters"].get("properties", {}))
    obligatoire = fiche["function"]["parameters"].get("required", [])
    print(f"  {fiche['function']['name']:<20} proprietes={proprietes} obligatoires={obligatoire}")
print(f"\n{len(catalogue_outils)} outil(s) expose(s) par le serveur, "
      f"{len(catalogue_outils)} transmis a l'agent (meme source : tools/list).")

CATALOGUE D'OUTILS TRANSMIS A L'AGENT
  transcrire_audio     proprietes=['chemin', 'langue'] obligatoires=['chemin']
  synthetiser_voix     proprietes=['texte', 'voix', 'langue', 'vitesse', 'nom_fichier'] obligatoires=['texte']
  analyser_audio       proprietes=['chemin'] obligatoires=['chemin']

3 outil(s) expose(s) par le serveur, 3 transmis a l'agent (meme source : tools/list).


In [9]:
# La batterie de consignes : l'agent doit choisir l'outil, on mesure ce qu'il choisit
from openai import OpenAI

# Les consignes sont en langue naturelle et nomment le fichier : c'est le
# contrat reel d'un agent (il ne connait pas le disque, on lui donne la cible).
TACHES = [
    {"consigne": f"Quel est le texte dit dans le fichier {CHEMIN_AUDIO} ? Transcris-le.",
     "attendu": "transcrire_audio", "genre": "transcription",
     "sequence": ["transcrire_audio"]},
    {"consigne": f"Combien de secondes dure le fichier {CHEMIN_AUDIO} ? Mesure sa duree.",
     "attendu": "analyser_audio", "genre": "duree",
     "sequence": ["analyser_audio"]},
    {"consigne": f"Quel est le niveau moyen du fichier {CHEMIN_AUDIO} en dBFS ?",
     "attendu": "analyser_audio", "genre": "niveau",
     "sequence": ["analyser_audio"]},
    {"consigne": "Synthese la phrase « Bonjour, voici le serveur MCP audio » avec la voix af_heart.",
     "attendu": "synthetiser_voix", "genre": "synthese",
     "sequence": ["synthetiser_voix"]},
    {"consigne": f"Transcris le fichier {CHEMIN_AUDIO} puis resume ce qui est dit.",
     "attendu": "transcrire_audio", "genre": "resume",
     "sequence": ["transcrire_audio"]},
    {"consigne": f"Ecris un fichier audio ou l'on dit « Le serveur audio repond present ».",
     "attendu": "synthetiser_voix", "genre": "synthese-ecriture",
     "sequence": ["synthetiser_voix"]},
    {"consigne": (f"Transcris le fichier {CHEMIN_AUDIO}, puis synthetise le texte obtenu "
                  "avec la voix af_heart dans un fichier nomme chaine.wav."),
     "attendu": "transcrire_audio", "genre": "chaine-deux-outils",
     "sequence": ["transcrire_audio", "synthetiser_voix"]},
]
TACHES = TACHES[:nb_taches]

SYSTEME_AGENT = (
    "Tu disposes d'outils audio : transcription, synthese vocale, analyse de fichiers audio. "
    "Appelle l'outil qui repond exactement a la question ; n'invente jamais la reponse, "
    f"appelle l'outil. Le fichier audio disponible est : {CHEMIN_AUDIO}. "
    "Apres l'appel de l'outil, reponds en francais avec la valeur mesuree."
)

client_agent = OpenAI(base_url=base_vllm, api_key=cle_vllm)


def classer_verdict(sequence_attendue: list, appels: list) -> str:
    """Verdict de la sequence d'outils attendue.

    - `AUCUN_OUTIL`   : l'agent a repondu sans appeler d'outil ;
    - `CORRECT`       : les outils appelés sont ceux attendus, dans l'ordre ;
    - `INCOMPLET`     : la sequence a ete commencee mais pas menee a son terme ;
    - `MAUVAIS_OUTIL` : un autre outil a ete choisi.
    """
    noms = [appel["outil"] for appel in appels]
    if not noms:
        return "AUCUN_OUTIL"
    if noms[:len(sequence_attendue)] == list(sequence_attendue):
        return "CORRECT" if len(noms) >= len(sequence_attendue) else "INCOMPLET"
    return "MAUVAIS_OUTIL"


def executer_consigne(consigne: str) -> dict:
    """Fait tourner l'agent sur une consigne et rend la trace de ses appels."""
    messages = [{"role": "system", "content": SYSTEME_AGENT},
                {"role": "user", "content": consigne}]
    appels = []
    debut = time.perf_counter()
    for _ in range(max_tours_agent):
        reponse = client_agent.chat.completions.create(
            model=vllm_model, messages=messages, tools=catalogue_outils,
            tool_choice="auto", temperature=temperature_agent, timeout=600)
        message = reponse.choices[0].message
        if not message.tool_calls:
            return {"reponse": message.content, "appels": appels,
                    "total_s": round(time.perf_counter() - debut, 2)}
        messages.append({"role": "assistant", "content": message.content or "",
                         "tool_calls": [{"id": c.id, "type": "function",
                                         "function": {"name": c.function.name,
                                                      "arguments": c.function.arguments}}
                                        for c in message.tool_calls]})
        for appel in message.tool_calls:
            arguments = json.loads(appel.function.arguments or "{}")
            t0 = time.perf_counter()
            resultat = session_mcp.appeler(appel.function.name, arguments)
            duree_appel = time.perf_counter() - t0
            texte = resultat.content[0].text
            erreur = getattr(resultat, "isError", False)
            appels.append({"outil": appel.function.name, "arguments": arguments,
                           "duree_s": round(duree_appel, 2), "erreur": bool(erreur),
                           "charge": texte[:600]})
            messages.append({"role": "tool", "tool_call_id": appel.id,
                             "content": texte[:2000]})
    return {"reponse": None, "appels": appels,
            "total_s": round(time.perf_counter() - debut, 2)}


print(f"MOTEUR DE L'AGENT : {vllm_model} sur {base_vllm}")
print(f"consignes : {len(TACHES)} | repetitions : {nb_repetitions}")
if not moteur_chat_joignable:
    print("MOTEUR DE CHAT INJOIGNABLE : la batterie ne peut pas etre executee.")

MOTEUR DE L'AGENT : qwen3.6-35b-a3b sur http://127.0.0.1:5002/v1
consignes : 7 | repetitions : 2


In [10]:
# Execution de la batterie : chaque consigne est repetee, chaque appel est horodate
resultats_batterie = []

if moteur_chat_joignable:
    for repetition in range(nb_repetitions):
        for tache in TACHES:
            trace = executer_consigne(tache["consigne"])
            premier = trace["appels"][0]["outil"] if trace["appels"] else None
            sequence_obtenue = [appel["outil"] for appel in trace["appels"]]
            resultats_batterie.append({
                "repetition": repetition,
                "genre": tache["genre"],
                "consigne": tache["consigne"],
                "attendu": tache["attendu"],
                "obtenu": premier,
                "sequence_attendue": tache["sequence"],
                "sequence_obtenue": sequence_obtenue,
                "verdict": classer_verdict(tache["sequence"], trace["appels"]),
                "nb_appels": len(trace["appels"]),
                "enchainement": len(trace["appels"]) > 1,
                "latence_outil_s": round(trace["appels"][0]["duree_s"], 2) if trace["appels"] else None,
                "latences_outils_s": [a["duree_s"] for a in trace["appels"]],
                "bout_en_bout_s": trace["total_s"],
                "erreurs": sum(1 for a in trace["appels"] if a["erreur"]),
                "reponse": (trace["reponse"] or "")[:300],
            })
            print(f"[rep {repetition}] {tache['genre']:<20} "
                  f"attendu={'->'.join(tache['sequence']):<38} "
                  f"obtenu={'->'.join(sequence_obtenue) or '(aucun)':<38} "
                  f"{resultats_batterie[-1]['verdict']:<13} | {trace['total_s']} s", flush=True)
    print(f"\nlignes collectees : {len(resultats_batterie)}")
else:
    print("Batterie non executee : le moteur de chat est injoignable.")

[rep 0] transcription        attendu=transcrire_audio                       obtenu=transcrire_audio                       CORRECT       | 13.62 s


[rep 0] duree                attendu=analyser_audio                         obtenu=analyser_audio                         CORRECT       | 13.89 s


[rep 0] niveau               attendu=analyser_audio                         obtenu=analyser_audio                         CORRECT       | 7.6 s


[rep 0] synthese             attendu=synthetiser_voix                       obtenu=synthetiser_voix                       CORRECT       | 13.02 s


[rep 0] resume               attendu=transcrire_audio                       obtenu=transcrire_audio                       CORRECT       | 10.22 s


[rep 0] synthese-ecriture    attendu=synthetiser_voix                       obtenu=synthetiser_voix                       CORRECT       | 12.23 s


... (5 outputs left out)

[rep 1] resume               attendu=transcrire_audio                       obtenu=transcrire_audio                       CORRECT       | 11.53 s


[rep 1] synthese-ecriture    attendu=synthetiser_voix                       obtenu=synthetiser_voix                       CORRECT       | 8.49 s


[rep 1] chaine-deux-outils   attendu=transcrire_audio->synthetiser_voix     obtenu=transcrire_audio->synthetiser_voix     CORRECT       | 53.46 s



lignes collectees : 14


## 5. La table de mesure

Les chiffres ci-dessous sortent de l'exécution : aucun n'est écrit à la main. Trois familles de mesures :

- **le choix d'outil** — la part des consignes où le premier outil appelé est celui qu'on attendait, et l'intervalle de confiance à 95 % (Wilson) de cette proportion ;
- **la latence de l'outil** — le temps que le moteur a mis, mesuré côté client autour de l'appel MCP ;
- **le temps bout en bout** — de la consigne à la réponse textuelle de l'agent : ilIncludes la latence du modèle, qui n'est pas un moteur audio.

Lire le tableau, c'est lire une **decomposition** : le temps de bout en bout n'est pas la somme des temps d'outils, parce que le modèle parle aussi.

In [11]:
# Table de mesure : le choix d'outil, les latences, le bout en bout
def intervalle_wilson(succes: int, total: int, z: float = 1.96) -> tuple:
    """Intervalle de confiance a 95 % d'une proportion (Wilson)."""
    if total == 0:
        return (0.0, 0.0)
    proportion = succes / total
    denominateur = 1 + z ** 2 / total
    centre = (proportion + z ** 2 / (2 * total)) / denominateur
    ecart = z * ((proportion * (1 - proportion) / total + z ** 2 / (4 * total ** 2)) ** 0.5) / denominateur
    return (max(0.0, centre - ecart), min(1.0, centre + ecart))


print("TABLE DE MESURE")
if resultats_batterie:
    print(f"{'genre':<21}{'attendu':<39}{'obtenu':<39}{'verdict':<13}"
          f"{'appels':>7}{'outil_s':>10}{'bout_s':>9}")
    print("-" * 138)
    for ligne in resultats_batterie:
        attendu = "->".join(ligne["sequence_attendue"])
        obtenu = "->".join(ligne["sequence_obtenue"]) or "(aucun)"
        print(f"{ligne['genre']:<21}{attendu:<39}{obtenu:<39}"
              f"{ligne['verdict']:<13}{ligne['nb_appels']:>7}"
              f"{str(ligne['latence_outil_s']):>10}{ligne['bout_en_bout_s']:>9}")
    print("-" * 138)

    total = len(resultats_batterie)
    corrects = sum(1 for ligne in resultats_batterie if ligne["verdict"] == "CORRECT")
    bas, haut = intervalle_wilson(corrects, total)
    print(f"\nCHOIX D'OUTIL : {corrects}/{total} corrects "
          f"({100 * corrects / total:.1f} %) — IC95 [{100 * bas:.1f} % ; {100 * haut:.1f} %]")

    par_genre = {}
    for ligne in resultats_batterie:
        par_genre.setdefault(ligne["genre"], []).append(ligne)
    print("\nPAR GENRE DE CONSIGNE")
    for genre, lignes in par_genre.items():
        justes = sum(1 for l in lignes if l["verdict"] == "CORRECT")
        print(f"  {genre:<18} {justes}/{len(lignes)} corrects")
else:
    print("Aucune mesure : le moteur de chat etait injoignable.")

latences_outils = [l["latence_outil_s"] for l in resultats_batterie if l["latence_outil_s"]]
latences_bout = [l["bout_en_bout_s"] for l in resultats_batterie]
if latences_outils:
    print(f"\nLATENCE D'UN OUTIL (client, autour de l'appel MCP)")
    print(f"  min {min(latences_outils):.2f} s | mediane {sorted(latences_outils)[len(latences_outils) // 2]:.2f} s"
          f" | max {max(latences_outils):.2f} s")
if latences_bout:
    print(f"BOUT EN BOUT (consigne -> reponse de l'agent)")
    print(f"  min {min(latences_bout):.2f} s | mediane {sorted(latences_bout)[len(latences_bout) // 2]:.2f} s"
          f" | max {max(latences_bout):.2f} s")
    part_modele = [l["bout_en_bout_s"] - sum(l["latences_outils_s"]) for l in resultats_batterie]
    print(f"  part hors outils (modele + transport) : "
          f"mediane {sorted(part_modele)[len(part_modele) // 2]:.2f} s")
if resultats_batterie:
    total_appels = sum(l["nb_appels"] for l in resultats_batterie)
    total_erreurs = sum(l["erreurs"] for l in resultats_batterie)
    enchainements = sum(1 for l in resultats_batterie if l["enchainement"])
    print(f"\nAPPELS : {total_appels} au total | erreurs {total_erreurs} "
          f"| consignes a enchainement {enchainements}/{len(resultats_batterie)}")

TABLE DE MESURE
genre                attendu                                obtenu                                 verdict       appels   outil_s   bout_s
------------------------------------------------------------------------------------------------------------------------------------------
transcription        transcrire_audio                       transcrire_audio                       CORRECT            1       3.3    13.62
duree                analyser_audio                         analyser_audio                         CORRECT            1      0.01    13.89
niveau               analyser_audio                         analyser_audio                         CORRECT            1      0.01      7.6
synthese             synthetiser_voix                       synthetiser_voix                       CORRECT            1      1.87    13.02
resume               transcrire_audio                       transcrire_audio                       CORRECT            1      2.67    10.22
synthese-ec

### Lecture du résultat

Les chiffres ci-dessus sont ceux de l'exécution affichée juste avant ; ils changent d'une machine à l'autre (charge CPU, taille du modèle servi), mais **leur ordre de grandeur** et **leur lecture** restent valables.

**Le choix d'outil est la mesure qui porte la démonstration.** Sur les 14 consignes exécutées (7 consignes × 2 répétitions), l'agent a choisi l'outil attendu **14 fois** — un taux de 100 %, dont l'intervalle de confiance à 95 % va de **78,5 % à 100 %** : l'incertitude réelle de l'échantillon reste large, et c'est pourquoi la table affiche l'intervalle et non un pourcentage nu. Aucune consigne n'a été servie sans appel d'outil (`AUCUN_OUTIL`), et les 16 appels d'outils n'ont produit **aucune erreur**.

**L'enchaînement fonctionne.** La consigne « transcris puis synthétise » a déclenché **deux** appels, dans l'ordre (`transcrire_audio` puis `synthetiser_voix`) : c'est le seul genre où l'agent a fait deux outils, et il les a faits dans le bon ordre.

**Les trois moteurs n'ont pas le même coût.** Dans la table, l'analyse d'un fichier de 8,7 s apparaît à **0,01 s** côté client (le moteur lui-même annonce 2 ms), la transcription entre **2,67 s et 5,42 s**, la synthèse entre **0,78 s et 1,87 s** ; la médiane de tous les appels est de **1,87 s**. Le coût du **transport MCP** (aller-retour stdio, encodage JSON-RPC) est mesuré séparément, sur l'outil le moins cher : **6,7 ms**. Il est donc négligeable devant le calcul — et c'est la seule façon de le dire honnêtement : mesuré sur la transcription, le même écart se noierait dans la variation du moteur d'un appel à l'autre.

**Le temps bout en bout n'est pas la somme des temps d'outils.** Médiane : **13,02 s** de la consigne à la réponse de l'agent, dont **11,12 s** ne sont pas des outils — c'est le modèle qui décide, appelle, reçoit, relit et formule. Un utilisateur perçoit les 13 s, pas les 1,87 s de l'outil. Le maximum observé (53,46 s) trahit une machine partagée : c'est la queue, pas la médiane, qui parle de la saturation.

**Une limite de la transcription, mesurée.** Le rappel des mots de la référence est de **13/19 (68 %)** : le début du texte est transcrit correctement, mais l'énumération finale (« transcrire, synthétiser et analyser ») est rendue approximativement. Deux appels du même moteur sur le même fichier n'ont pas produit le même texte : la transcription n'est donc pas reproductible au caractère près ici, et la mesure porte sur un rappel de mots plutôt que sur une égalité de chaînes. C'est la limite réelle du modèle `small` sur CPU, pas un défaut du protocole — les deux chemins, direct et outil, donnent le même texte.

Trois limites plus larges sont à garder en tête : un seul moteur de chat a été mesuré, sur sept consignes en français ; la répétition par consigne est faible (deux), et l'intervalle de Wilson rend cette incertitude visible au lieu de la masquer ; enfin l'agent ne « comprend » pas l'audio — il appelle un outil, qui appelle un moteur, et c'est précisément la valeur de passer par le protocole.

## 6. Exercice 1 — router une consigne vers le bon outil

Le carnet s'appuie sur un modèle pour choisir. Vous allez écrire la **règle déterministe** qui fait la même chose sans modèle : c'est la base d'un routage fiable, et la référence contre laquelle on juge l'agent.

À vous de :

1. écrire `choisir_outil(consigne, catalogue)` qui renvoie le nom de l'outil attendu pour une consigne donnée, en s'appuyant sur les mots-clés de la consigne et sur les descriptions du catalogue ;
2. l'appliquer aux consignes de la batterie et comparer avec l'outil choisi par l'agent (`resultats_batterie`).

Le catalogue est fourni : `session_mcp.outils` (nom, description, schéma d'arguments).

In [12]:
# Exercice 1 : routeur deterministe (a completer)
# TODO : implementer choisir_outil
# Indice : les descriptions des outils contiennent les mots qui distinguent
#          transcription / synthese / analyse ; le routage peut se faire sur
#          le nom de l'outil et sur les verbes de la consigne.


def choisir_outil(consigne: str, catalogue: list) -> str:
    """Renvoie le nom de l'outil qui repond a la consigne.

    :param consigne: la demande en langue naturelle ;
    :param catalogue: la liste des outils exposes (dicts avec 'name' et 'description') ;
    :return: le nom de l'outil choisi.
    """
    # TODO etudiant : remplacer par le routage reel
    print("Exercice a completer : choisir_outil")
    return None


# --- Verification : le routeur doit designer le meme outil que l'agent
if resultats_batterie:
    accords = 0
    vues = set()
    for ligne in resultats_batterie:
        if ligne["consigne"] in vues:
            continue
        vues.add(ligne["consigne"])
        prediction = choisir_outil(ligne["consigne"], session_mcp.outils)
        if prediction == ligne["obtenu"]:
            accords += 1
        print(f"  {ligne['genre']:<21} agent={ligne['obtenu']} routeur={prediction} "
              f"{'accord' if prediction == ligne['obtenu'] else 'divergence'}")
    print(f"\naccords routeur / agent : {accords}/{len(vues)} consignes distinctes")
else:
    print("Batterie non executee : rien a comparer.")

Exercice a completer : choisir_outil
  transcription         agent=transcrire_audio routeur=None divergence
Exercice a completer : choisir_outil
  duree                 agent=analyser_audio routeur=None divergence
Exercice a completer : choisir_outil
  niveau                agent=analyser_audio routeur=None divergence
Exercice a completer : choisir_outil
  synthese              agent=synthetiser_voix routeur=None divergence
Exercice a completer : choisir_outil
  resume                agent=transcrire_audio routeur=None divergence
Exercice a completer : choisir_outil
  synthese-ecriture     agent=synthetiser_voix routeur=None divergence
Exercice a completer : choisir_outil
  chaine-deux-outils    agent=transcrire_audio routeur=None divergence

accords routeur / agent : 0/7 consignes distinctes


## 7. Exercice 2 — calculer le taux de bon choix et son intervalle

La proportion « 4 sur 6 » a une lecture, mais l'incertitude fait partie du résultat : sur six essais, un écart de un point change beaucoup la conclusion. L'intervalle de Wilson est la façon simple de le dire.

À vous de :

1. écrire `taux_choix_correct(résultats, genre=None)` qui renvoie la proportion de choix corrects, globalement ou pour un genre de consigne donné ;
2. écrire `intervalle_wilson(succès, total)` — la formule est rappelée dans la cellule de mesure, mais **écrivez-la** : c'est elle qui distingue un taux de 60 % d'un taux de 100 % ;
3. afficher le taux et l'intervalle pour chaque genre.

In [13]:
# Exercice 2 : taux de bon choix et intervalle de confiance (a completer)
# TODO : implementer taux_choix_correct et intervalle_wilson


def taux_choix_correct(resultats: list, genre: str | None = None) -> float:
    """Proportion de consignes ou l'agent a choisi l'outil attendu.

    :param resultats: les lignes de `resultats_batterie` ;
    :param genre: restreint la mesure a un genre de consigne ; None = tout ;
    :return: la proportion de choix corrects, entre 0 et 1.
    """
    # TODO etudiant : remplacer par le calcul reel
    print("Exercice a completer : taux_choix_correct")
    return None


def intervalle_wilson(succes: int, total: int, z: float = 1.96) -> tuple:
    """Intervalle de confiance a 95 % d'une proportion (methode de Wilson).

    :param succes: nombre de succes ;
    :param total: nombre d'essais ;
    :param z: quantile normal (1.96 pour 95 %) ;
    :return: le couple (borne basse, borne haute).
    """
    # TODO etudiant : remplacer par la formule
    print("Exercice a completer : intervalle_wilson")
    return (None, None)


# --- Affichage attendu une fois l'exercice resolu
if resultats_batterie:
    genres = sorted({ligne["genre"] for ligne in resultats_batterie})
    for genre in genres + [None]:
        taux = taux_choix_correct(resultats_batterie, genre)
        bas, haut = intervalle_wilson(round((taux or 0) * len(resultats_batterie)),
                                      len(resultats_batterie))
        libelle = genre or "tous genres"
        print(f"  {libelle:<18} taux={taux} IC95=[{bas}, {haut}]")
else:
    print("Batterie non executee : rien a calculer.")

Exercice a completer : taux_choix_correct
Exercice a completer : intervalle_wilson
  chaine-deux-outils taux=None IC95=[None, None]
Exercice a completer : taux_choix_correct
Exercice a completer : intervalle_wilson
  duree              taux=None IC95=[None, None]
Exercice a completer : taux_choix_correct
Exercice a completer : intervalle_wilson
  niveau             taux=None IC95=[None, None]
Exercice a completer : taux_choix_correct
Exercice a completer : intervalle_wilson
  resume             taux=None IC95=[None, None]
Exercice a completer : taux_choix_correct
Exercice a completer : intervalle_wilson
  synthese           taux=None IC95=[None, None]
Exercice a completer : taux_choix_correct
Exercice a completer : intervalle_wilson
  synthese-ecriture  taux=None IC95=[None, None]
Exercice a completer : taux_choix_correct
Exercice a completer : intervalle_wilson
  transcription      taux=None IC95=[None, None]
Exercice a completer : taux_choix_correct
Exercice a completer : intervalle_

## 8. Exercice 3 — un enchaînement de deux outils

L'issue demande des **enchaînements corrects** : une consigne qui demande deux choses doit voir le carnet appeler deux outils, dans le bon ordre, le second recevant le résultat du premier.

À vous de :

1. écrire `pipeline_resume_podcast(chemin_audio, chemin_sortie)` qui transcrit un fichier, puis fait **synthétiser** par Kokoro le résumé de la transcription, et qui rend les deux étapes (texte, chemin du WAV, durées) ;
2. l'exécuter sur l'audio de travail et vérifier que le WAV de sortie existe et qu'il n'est pas vide.

Le squelette est donné ; les deux appels d'outils doivent passer par `session_mcp.appeler(...)` — **pas** par un appel direct au moteur : l'exercice porte sur l'enchaînement d'outils, pas sur les moteurs.

In [14]:
# Exercice 3 : pipeline transcription -> resume synthetise (a completer)
# TODO : implementer pipeline_resume_podcast


def pipeline_resume_podcast(chemin_audio: str, chemin_sortie: str) -> dict:
    """Transcrit un audio, puis synthetise le resume de la transcription.

    :param chemin_audio: fichier audio a transcrire ;
    :param chemin_sortie: nom du WAV de sortie (sans extension) ;
    :return: un dict avec le texte transcrit, le chemin du WAV produit et
        la duree de chaque etape.
    """
    # TODO etudiant : appeler transcrire_audio puis synthetiser_voix par le canal MCP
    print("Exercice a completer : pipeline_resume_podcast")
    return {"texte": None, "chemin_sortie": None, "durees_s": {}}


# --- Verification attendue une fois l'exercice resolu
sortie_exercice = pipeline_resume_podcast(str(CHEMIN_AUDIO), "resume_podcast")
print(f"transcription : {sortie_exercice['texte']}")
print(f"sortie       : {sortie_exercice['chemin_sortie']}")
print(f"durees       : {sortie_exercice['durees_s']}")
if sortie_exercice["chemin_sortie"]:
    chemin_wav = Path(sortie_exercice["chemin_sortie"])
    print(f"fichier present : {chemin_wav.exists()} "
          f"({chemin_wav.stat().st_size if chemin_wav.exists() else 0} octets)")

Exercice a completer : pipeline_resume_podcast
transcription : None
sortie       : None
durees       : {}


## 9. Conclusion et suite

Ce carnet a branche trois moteurs audio de la série derrière un **serveur MCP réel**, et a fait choisir ses outils a un modèle reel sur des consignes en langue naturelle. Trois choses sont mesurées, pas affirmées : le choix d'outil, la latence par appel, le temps bout en bout.

Ce que la démonstration ne dit pas :

- un seul moteur de chat a été mesuré, sur une dizaine de consignes en francais ;
- l'agent ne « comprend » pas l'audio : il appelle un outil, qui appelle un moteur — c'est précisément la valeur de passer par le protocole ;
- la latence dépend de la charge de la machine ; les chiffres sont ceux de cette exécution.

**Suite de la série** : [02-5-Multi-Model-TTS-Gateway](../../02-Advanced/02-5-Multi-Model-TTS-Gateway.ipynb) expose plusieurs moteurs de synthèse derrière une même interface — la même logique, appliquée au seul TTS.

**Navigation** : [<< 03-3](03-3-Realtime-Voice-API.ipynb) | [Index](../README.md) | [Index Audio](../README.md)

In [15]:
# Fermeture : le serveur MCP est arrete proprement
if 'session_mcp' in globals():
    session_mcp.fermer()
    print("Serveur MCP arrete.")

print("\nRESUME DE LA SESSION")
print(f"  serveur MCP     : {session_mcp.info.serverInfo.name} "
      f"(protocole {session_mcp.info.protocolVersion})" if 'session_mcp' in globals() else "  serveur MCP : absent")
print(f"  outils exposes  : {len(session_mcp.outils) if 'session_mcp' in globals() else 0}")
print(f"  lignes mesurees : {len(resultats_batterie)}")
print(f"  sorties         : {OUTPUT_DIR}")

Serveur MCP arrete.

RESUME DE LA SESSION
  serveur MCP     : serveur-mcp-audio (protocole 2025-11-25)
  outils exposes  : 3
  lignes mesurees : 14
  sorties         : D:\CoursIA-wt\audio-mcp\MyIA.AI.Notebooks\GenAI\outputs\audio\mcp
